In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

In [2]:
ARQUIVO = "projeto_previsao_atraso_fornecedor.xlsx"

abas = pd.read_excel(ARQUIVO, sheet_name=None)

for nome, tabela in abas.items():
    print(f"{nome:<22} {tabela.shape}")

LEIA-ME                (22, 3)
PEDIDOS_ML             (1500, 30)
RESULTADO_POS_ENTREGA  (1500, 6)
FORNECEDORES           (20, 9)
PRODUTOS               (60, 5)
DICIONARIO_DADOS       (30, 4)


In [3]:
df = abas["PEDIDOS_ML"].copy()

df.head()

,ID_Pedido,Data_Pedido,ID_Fornecedor,Fornecedor,Perfil_Fornecedor,ID_Produto,Produto,Categoria_Produto,ID_Filial,Filial,Regiao_Destino,Canal,Transportadora,Modal,Prioridade,Quantidade,Preco_Unitario,Valor_Pedido,Peso_Total_kg,Distancia_km,Prazo_Contratado_Dias,Lead_Time_Medio_Historico,Desvio_Lead_Time_Historico,Taxa_Atraso_Historica_90d,Pedidos_90d_Fornecedor,Nota_Fornecedor,Indice_Urgencia,Cobertura_Estoque_Dias,Pedidos_Abertos_Fornecedor,Atrasou
0,PED00001,2025-05-10,FOR001,AutoPartes Sul,Excelente,PRD0046,Arrefecimento - Item 046,Arrefecimento,FIL03,Belo Horizonte,Sudeste,Distribuição Direta,Sul Cargo,Rodoviário,Normal,256,416.67,106667.52,1272.32,31,2,3,3.02,0.067,75,61,56,8.8,0,0
1,PED00002,2025-06-02,FOR003,Freios Master,Excelente,PRD0026,Suspensão - Item 026,Suspensão,FIL08,Manaus,Norte,E-commerce,TransLog,Rodoviário,Normal,160,801.58,128252.80,2865.60,2652,9,11,1.40,0.087,179,81,77,15.1,4,0
2,PED00003,2025-04-12,FOR015,Prime Automotive,Regular,PRD0020,Elétrica - Item 020,Elétrica,FIL07,Salvador,Nordeste,Distribuição Direta,Sul Cargo,Rodoviário,Normal,111,234.09,25983.99,300.81,1366,12,9,1.60,0.266,185,75,30,4.8,0,0
3,PED00004,2026-04-10,FOR016,TecnoMolas,Boa,PRD0019,Motor - Item 019,Motor,FIL01,São Paulo,Sudeste,Filial,Expresso Brasil,Rodoviário,Alta,41,1597.17,65483.97,890.52,144,13,10,1.99,0.181,157,64,80,14.9,1,0
4,PED00005,2026-08-06,FOR009,PlastCar,Boa,PRD0053,Transmissão - Item 053,Transmissão,FIL03,Belo Horizonte,Sudeste,Filial,TransLog,Rodoviário,Normal,152,2097.46,318813.92,3230.00,177,5,4,0.94,0.146,114,60,51,23.6,4,0


In [4]:
print("Linhas:", df.shape[0])
print("Colunas:", df.shape[1])
print()
df.info()

Linhas: 1500
Colunas: 30

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 30 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   ID_Pedido                   1500 non-null   object        
 1   Data_Pedido                 1500 non-null   datetime64[ns]
 2   ID_Fornecedor               1500 non-null   object        
 3   Fornecedor                  1500 non-null   object        
 4   Perfil_Fornecedor           1500 non-null   object        
 5   ID_Produto                  1500 non-null   object        
 6   Produto                     1500 non-null   object        
 7   Categoria_Produto           1500 non-null   object        
 8   ID_Filial                   1500 non-null   object        
 9   Filial                      1500 non-null   object        
 10  Regiao_Destino              1500 non-null   object        
 11  Canal                       15

In [5]:
colunas_numericas = df.select_dtypes(include="number").columns.tolist()
colunas_categoricas = df.select_dtypes(include=["object", "string"]).columns.tolist()
colunas_data = df.select_dtypes(include="datetime").columns.tolist()

print("Numéricas  :", len(colunas_numericas), colunas_numericas)
print("Categóricas:", len(colunas_categoricas), colunas_categoricas)
print("Datas      :", len(colunas_data), colunas_data)

Numéricas  : 15 ['Quantidade', 'Preco_Unitario', 'Valor_Pedido', 'Peso_Total_kg', 'Distancia_km', 'Prazo_Contratado_Dias', 'Lead_Time_Medio_Historico', 'Desvio_Lead_Time_Historico', 'Taxa_Atraso_Historica_90d', 'Pedidos_90d_Fornecedor', 'Nota_Fornecedor', 'Indice_Urgencia', 'Cobertura_Estoque_Dias', 'Pedidos_Abertos_Fornecedor', 'Atrasou']
Categóricas: 14 ['ID_Pedido', 'ID_Fornecedor', 'Fornecedor', 'Perfil_Fornecedor', 'ID_Produto', 'Produto', 'Categoria_Produto', 'ID_Filial', 'Filial', 'Regiao_Destino', 'Canal', 'Transportadora', 'Modal', 'Prioridade']
Datas      : 1 ['Data_Pedido']


In [6]:
print("Valores ausentes por coluna:")
print(df.isna().sum().sum(), "no total")

print("Linhas totalmente duplicadas:", df.duplicated().sum())
print("ID_Pedido repetidos:", df["ID_Pedido"].duplicated().sum())

Valores ausentes por coluna:
0 no total
Linhas totalmente duplicadas: 0
ID_Pedido repetidos: 0


In [8]:
contagem = df["Atrasou"].value_counts()
percentual = df["Atrasou"].value_counts(normalize=True) * 100

resumo = pd.DataFrame({
    "Quantidade": contagem,
    "Percentual (%)": percentual.round(1)
})
resumo.index = resumo.index.map({0: "0 - Não atrasou", 1: "1 - Atrasou"})
resumo.index.name = "Atrasou"

print(resumo)
print()
print("Período:", df["Data_Pedido"].min().date(), "até", df["Data_Pedido"].max().date())

                 Quantidade  Percentual (%)
Atrasou                                    
0 - Não atrasou         975            65.0
1 - Atrasou             525            35.0

Período: 2025-01-01 até 2026-09-30
